# BirdCLEF+ 2025 - Inference profiling (v2 baseline)

This is a profiling copy of our team's baseline inference notebook `sc4000-birdclef-m1-infer-v2`. The model, mel settings, windowing and batching are unchanged. It runs on CPU with the internet off, on up to `PROFILE_N_FILES` (default 700) files from `train_soundscapes`, and records the wall-clock time for audio decode, mel computation and model forward in `timing_profile.json`.

`PROFILE_NUM_WORKERS` defaults to 1 (single-process). The baseline notebook computes mels in `Pool(os.cpu_count())`, so set `PROFILE_NUM_WORKERS` to the CPU count to reproduce that. With more than one worker, the decode and mel values are summed across workers, so use `preprocess_elapsed` for the elapsed time.

`submission.csv` is written only for comparison with the baseline's dry run. It covers training soundscapes and is not a competition submission.


In [ ]:
import os, glob, json, time
NOTEBOOK_T0 = time.perf_counter()
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import timm

def find_data_dir():
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

def find_model_dir():
    hits = glob.glob("/kaggle/input/**/model.pth", recursive=True)
    return Path(hits[0]).parent if hits else Path("../../outputs/jamie_train_baseline")

DATA, MODEL_DIR = find_data_dir(), find_model_dir()
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
torch.set_num_threads(os.cpu_count())
print("Data:", DATA, "| model:", MODEL_DIR, "| CPUs:", os.cpu_count())

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print(sample_sub.iloc[:3, :4])

In [ ]:
N_FILES = int(os.environ.get("PROFILE_N_FILES", 700))
NUM_WORKERS = int(os.environ.get("PROFILE_NUM_WORKERS", os.cpu_count()))

available = sorted((DATA / "train_soundscapes").glob("*.ogg"))
test_files = available[:N_FILES]
if not test_files:
    raise RuntimeError(f"No files to profile: {len(available)} available, PROFILE_N_FILES={N_FILES}")
if len(test_files) < N_FILES:
    print(f"WARNING: requested {N_FILES} soundscapes but only {len(test_files)} exist; profiling those")
print(f"Profiling {len(test_files)} soundscapes with {NUM_WORKERS} worker(s)")
SETUP_S = time.perf_counter() - NOTEBOOK_T0


## Mel spectrograms (identical settings to training, read from `cfg.json`)

In [ ]:
WIN = CFG["sr"] * CFG["duration"]

def to_mel(y):
    m = librosa.feature.melspectrogram(
        y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    db = librosa.power_to_db(m, ref=np.max, top_db=80)
    return ((db + 80) / 80).astype(np.float32)

def soundscape_windows(path):
    t0 = time.perf_counter()
    y, _ = sf.read(path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    t1 = time.perf_counter()
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    mels = np.stack([to_mel(y[i * WIN:(i + 1) * WIN]) for i in range(n_win)])
    t2 = time.perf_counter()
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{path.stem}_{(i + 1) * CFG['duration']}" for i in range(n_win)]
    return row_ids, mels, t1 - t0, t2 - t1

PIPELINE_T0 = time.perf_counter()
if NUM_WORKERS > 1:
    with Pool(NUM_WORKERS) as pool:
        results = pool.map(soundscape_windows, test_files)
else:
    results = [soundscape_windows(p) for p in test_files]
row_ids = [r for ids, *_ in results for r in ids]
X = np.concatenate([m for _, m, *_ in results])
DECODE_S = sum(r[2] for r in results)
MEL_S = sum(r[3] for r in results)
PREPROCESS_ELAPSED_S = time.perf_counter() - PIPELINE_T0
print("Windows:", X.shape, f"in {PREPROCESS_ELAPSED_S:.1f}s")

## Predict

In [ ]:
t0 = time.perf_counter()
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
model.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
model.eval()
MODEL_LOAD_S = time.perf_counter() - t0

MODEL_FORWARD_S = 0.0; preds = []
with torch.inference_mode():
    for i in range(0, len(X), 64):
        xb = torch.from_numpy(X[i:i + 64])[:, None]
        t0 = time.perf_counter()
        preds.append(torch.sigmoid(model(xb)).numpy())
        MODEL_FORWARD_S += time.perf_counter() - t0
preds = np.concatenate(preds)
print("Predictions:", preds.shape, f"in {MODEL_FORWARD_S:.1f}s")

In [ ]:
# total_pipeline excludes setup and model loading, which do not scale with file count.
now = time.perf_counter()
n = len(test_files)
stages = {
    "setup": SETUP_S,
    "model_load": MODEL_LOAD_S,
    "decode": DECODE_S,
    "mel": MEL_S,
    "preprocess_elapsed": PREPROCESS_ELAPSED_S,
    "model_forward": MODEL_FORWARD_S,
    "total_pipeline": now - PIPELINE_T0 - MODEL_LOAD_S,
    "total_notebook": now - NOTEBOOK_T0,
}
per_file = ["decode", "mel", "preprocess_elapsed", "model_forward", "total_pipeline"]
profile = {
    "n_files_requested": N_FILES,
    "n_files_used": n,
    "n_windows": len(X),
    "num_workers": NUM_WORKERS,
    "torch_threads": torch.get_num_threads(),
    "cpu_count": os.cpu_count(),
    "batch_size": 64,
    "stages_s": {k: round(v, 4) for k, v in stages.items()},
    "per_file_mean_s": {k: round(stages[k] / n, 6) for k in per_file},
}
with open("timing_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print(json.dumps(profile, indent=2))


In [ ]:
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.iloc[:5, :6]